In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import time, os


In [4]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# 1️⃣ Define transforms (resize, normalize, etc.)
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # resize all images to same size
    transforms.ToTensor(),          # convert to tensor
    transforms.Normalize((0.5,), (0.5,))  # normalize pixel values
])

# 2️⃣ Load the dataset
data_dir = r"D:\Project\Waste management\data\train"  # path to your train folder
train_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# 3️⃣ Split train into train + validation
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

# 4️⃣ Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))


Train samples: 1615
Validation samples: 404


In [5]:
import torch.nn as nn
import torch.nn.functional as F

# Define a simple CNN architecture
class WasteClassifierCNN(nn.Module):
    def __init__(self, num_classes=6):  # TrashNet has 6 classes
        super(WasteClassifierCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 32 * 32, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # (3,128,128) -> (32,64,64)
        x = self.pool(F.relu(self.conv2(x)))  # (32,64,64) -> (64,32,32)
        x = x.view(-1, 64 * 32 * 32)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = WasteClassifierCNN().to(device)

print(model)

WasteClassifierCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=65536, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=6, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [6]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [7]:
num_epochs = 10  # you can increase to 20–30 later
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {running_loss/len(train_loader):.4f}")


Epoch [1/10] - Loss: 1.6281
Epoch [2/10] - Loss: 1.2034
Epoch [3/10] - Loss: 0.9148
Epoch [4/10] - Loss: 0.7056
Epoch [5/10] - Loss: 0.4966
Epoch [6/10] - Loss: 0.3222
Epoch [7/10] - Loss: 0.2144
Epoch [8/10] - Loss: 0.1388
Epoch [9/10] - Loss: 0.1309
Epoch [10/10] - Loss: 0.0783


In [8]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100 * correct / total:.2f}%")


Validation Accuracy: 67.57%


In [11]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Test transform (should match validation transform)
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# Load the test dataset
test_dataset = datasets.ImageFolder(root='data/test', transform=test_transform)

# Create DataLoader for test dataset
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [12]:
# Step 6: Evaluate on Test Set
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"🧾 Test Accuracy: {100 * correct / total:.2f}%")


🧾 Test Accuracy: 26.97%
